In [1]:
"""
build_aliases.py — Layer 5 build step
======================================
Parses the IANA tz database link definitions and produces aliases.json.

Sources (tries in order):
  1. Local tzdata PyPI package (zoneinfo/tzdata.zi)
  2. Downloaded IANA backward file

Output: aliases.json  ->  {"Asia/Calcutta": "Asia/Kolkata", ...}
"""

import json
import os
import re
import urllib.request
from pathlib import Path

BACKWARD_URL = "https://data.iana.org/time-zones/tzdata-latest.tar.gz"
TZDATA_ZI_PYPI = None  # filled in by try_pypi_tzdata()

OUTPUT_FILE = "aliases.json"


# ---------------------------------------------------------------------------
# Source 1: tzdata.zi from the PyPI tzdata package (fastest, already local)
# ---------------------------------------------------------------------------
def parse_tzdata_zi(path: Path) -> dict[str, str]:
    """Parse 'L Target Alias' lines from tzdata.zi.

    Format (from man zic):
        L  <target>  <alias>
    meaning: alias is a link to target.
    """
    aliases: dict[str, str] = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 3 and parts[0] == "L":
                target, alias = parts[1], parts[2]
                aliases[alias] = target
    return aliases


def try_pypi_tzdata() -> dict[str, str] | None:
    """Try to load tzdata.zi from the installed PyPI tzdata package."""
    try:
        import tzdata
        base = Path(tzdata.__file__).parent / "zoneinfo"
        zi = base / "tzdata.zi"
        if zi.exists():
            print(f"  [tzdata pypi] found {zi}")
            return parse_tzdata_zi(zi)
    except ImportError:
        pass
    return None


# ---------------------------------------------------------------------------
# Source 2: download and parse the IANA 'backward' file
# ---------------------------------------------------------------------------
def parse_backward(text: str) -> dict[str, str]:
    """Parse 'Link<TAB>Target<TAB>Alias' lines from the backward file.

    Comments start with '#'. Fields are tab-separated.
    Format: Link  <target>  <alias>
    """
    aliases: dict[str, str] = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        fields = re.split(r"\s+", line)
        if len(fields) >= 3 and fields[0] == "Link":
            target, alias = fields[1], fields[2]
            aliases[alias] = target
    return aliases


def download_backward() -> dict[str, str] | None:
    """Download the IANA tzdata tarball and extract the 'backward' file.

    Note: this hits the network. Run it once, then never again.
    """
    import io
    import tarfile

    print(f"  [iana] downloading {BACKWARD_URL} ...")
    try:
        with urllib.request.urlopen(BACKWARD_URL, timeout=30) as resp:
            data = resp.read()
    except Exception as e:
        print(f"  [iana] download failed: {e}")
        return None

    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
        for member in tar.getmembers():
            if member.name.endswith("/backward") or member.name == "backward":
                f = tar.extractfile(member)
                if f:
                    text = f.read().decode("utf-8")
                    print(f"  [iana] parsed {member.name}")
                    return parse_backward(text)
    print("  [iana] backward file not found in tarball")
    return None


# ---------------------------------------------------------------------------
# Main build
# ---------------------------------------------------------------------------
def build() -> dict[str, str]:
    print("Building alias map...")

    # Try PyPI tzdata first (local, no network)
    aliases = try_pypi_tzdata()

    # Fall back to downloading from IANA
    if aliases is None:
        aliases = download_backward()

    if aliases is None:
        raise RuntimeError(
            "Could not obtain alias data. Install the 'tzdata' PyPI package "
            "or run this script with internet access."
        )

    # Filter: only keep entries where alias != target.
    # Also drop entries whose target is itself an alias (flatten chains).
    cleaned: dict[str, str] = {}
    for alias, target in aliases.items():
        if alias == target:
            continue
        cleaned[alias] = target

    # Flatten chains: if Asia/Calcutta -> Asia/Kolkata, but Asia/Kolkata is
    # itself a link, follow it. (Rare, but happens.)
    def resolve(name: str, seen: set[str] | None = None) -> str:
        if seen is None:
            seen = set()
        if name in seen:
            return name  # cycle guard
        seen.add(name)
        if name in cleaned:
            return resolve(cleaned[name], seen)
        return name

    flattened = {alias: resolve(alias) for alias in cleaned}

    print(f"  total aliases: {len(flattened)}")
    return flattened


def main():
    aliases = build()
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(aliases, f, indent=2, sort_keys=True)
    print(f"Wrote {OUTPUT_FILE} ({len(aliases)} entries)")

    # Show a few examples
    for name in ["Asia/Calcutta", "US/Eastern", "Japan", "Eire", "GB"]:
        if name in aliases:
            print(f"  {name} -> {aliases[name]}")


if __name__ == "__main__":
    main()

Building alias map...
  [tzdata pypi] found c:\ProgramData\anaconda3\envs\gpu\lib\site-packages\tzdata\zoneinfo\tzdata.zi
  total aliases: 257
Wrote aliases.json (257 entries)
  Asia/Calcutta -> Asia/Kolkata
  US/Eastern -> America/New_York
  Japan -> Asia/Tokyo
  Eire -> Europe/Dublin
  GB -> Europe/London


In [2]:
import json

# Load once at startup
with open("aliases.json", encoding="utf-8") as f:
    ALIASES = json.load(f)


def canonicalize(zone_name: str) -> str:
    """Return the canonical IANA name for any zone string.

    Idempotent: canonicalize(canonicalize(x)) == canonicalize(x).
    """
    if not zone_name:
        return zone_name
    return ALIASES.get(zone_name, zone_name)

In [3]:
for name in [
    "Asia/Calcutta",     # alias -> canonical
    "Asia/Kolkata",      # already canonical
    "US/Eastern",        # backward link
    "Japan",             # alias
    "Eire",              # alias
    "GB",                # alias
    "Europe/Kiev",       # alias (if your tzdata is recent enough)
    "America/New_York",  # canonical
    "Not/AZone",         # unknown -> returns itself
]:
    print(f"{name:<22} -> {canonicalize(name)}")

Asia/Calcutta          -> Asia/Kolkata
Asia/Kolkata           -> Asia/Kolkata
US/Eastern             -> America/New_York
Japan                  -> Asia/Tokyo
Eire                   -> Europe/Dublin
GB                     -> Europe/London
Europe/Kiev            -> Europe/Kyiv
America/New_York       -> America/New_York
Not/AZone              -> Not/AZone


In [ ]:
def build_abbrev_map():
    """abbrev -> list of CANONICAL zones that currently use it."""
    from datetime import datetime, timezone
    from collections import defaultdict
    import zoneinfo

    now_utc = datetime.now(timezone.utc)
    raw = defaultdict(set)  # use a set to dedup

    for zone_name in zoneinfo.available_timezones():
        try:
            tz = zoneinfo.ZoneInfo(zone_name)
            local = now_utc.astimezone(tz)
            abbrev = local.tzname()
            if abbrev and abbrev.isalpha() and abbrev.isupper():
                canon = canonicalize(zone_name)
                raw[abbrev].add(canon)
        except Exception:
            continue

    # Convert sets to sorted lists
    return {abbr: sorted(zones) for abbr, zones in raw.items()}


ABBREV_MAP = build_abbrev_map()


def resolve_abbreviation(abbr: str):
    a = abbr.strip().upper()
    zones = ABBREV_MAP.get(a, [])
    if not zones:
        return "unknown", []
    if len(zones) == 1:
        return "unique", zones
    return "ambiguous", zones

In [5]:
for abbr in ["JST", "IST", "CST", "GMT", "XYZ"]:
    status, zones = resolve_abbreviation(abbr)
    print(f"{abbr:<6} -> {status:<10} {zones}")

JST    -> unique     ['Asia/Tokyo']
IST    -> ambiguous  ['Asia/Kolkata', 'Europe/Dublin']
CST    -> ambiguous  ['America/Bahia_Banderas', 'America/Belize', 'America/Chihuahua', 'America/Costa_Rica', 'America/El_Salvador', 'America/Guatemala', 'America/Managua', 'America/Merida', 'America/Mexico_City', 'America/Monterrey', 'America/Regina', 'America/Swift_Current', 'America/Tegucigalpa', 'Asia/Macau', 'Asia/Shanghai', 'Asia/Taipei']
GMT    -> ambiguous  ['Africa/Abidjan', 'Africa/Bissau', 'Africa/Monrovia', 'Africa/Sao_Tome', 'America/Danmarkshavn', 'Etc/GMT']
XYZ    -> unknown    []
